# Evaluation of Deep Learning Models

This notebook evaluates trained multi-class segmentation models specifically inside a **boundary zone of interest** (e.g., within 10 pixels of the water/land boundary). The approach directly follows `evaluate_pixel_based_metrics.ipynb` for fair comparison.

## Evaluation Strategy

- For each patch patch in the test dataset
- Predict the multi-class mask using the DL model 
- Collapse GT multi-class to binary water/no-water (with ignore mask)
- Collapse Prediction to binary water/no-water
- Calculate IoU and Dice metrics excluding ignore pixels and constrained to the boundary zone
- Aggregate results into a DataFrame for analysis

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import utility functions from evaluation_utils
from notebooks.utils.evaluation_utils import (
    collapse_gt_to_binary, 
    create_boundary_zone_mask, 
    calculate_metrics
)

# Import Deep Learning utilities
from src.dl_models import get_model, CoastlineDataset

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully!")

## Configuration

Define paths, models, class mappings, boundary zone parameters and load datasets.

**Note:** PyTorch `CoastlineDataset` subtracts 1 from all classes mapping values from 1-5 to 0-4 (with 255 as ignore_index):
* Original 3 & 5 $\rightarrow$ Mapped 2 & 4 (Water classes)
* Original 2 & 4 $\rightarrow$ Mapped 1 & 3 (No-water / land classes)
* Original 1 & 255 $\rightarrow$ Mapped 0 & 255 (Ignored classes)

In [ ]:
# Target Study Areas and General setup
STUDY_NAMES = ["puntal", "salinas"]

# Base directories
BASE_DIR = Path("..") / "data"
TEST_NPY_DIR = BASE_DIR / "dl_npy" / "test"
MODELS_DIR = Path("..") / "results" / "models"

# Available Models to Evaluate
# For now, evaluate the unet with resnet34 backbone
EVAL_MODELS = [
    {
        "name": "unet_resnet34",
        "arch": "unet",
        "encoder": "resnet34",
        "filepath": MODELS_DIR / "best_model_unet_resnet34.pth"
    }
]

# Inference parameters
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
BATCH_SIZE = 1 # Keep 1 for patch-level inference tracking

# Boundary zone evaluation parameters
BOUNDARY_OFFSET_PIXELS = 10  # Symmetric offset around water/land boundary
MIN_BOUNDARY_PIXELS = 50     # Minimum pixels required in boundary zone to evaluate patch

# Class Mapping definitions using mapped target labels (from CoastlineDataset)
MAPPED_GT_CLASS_MAPPING = {
    'water_classes': [2, 4],      # original 3(water), 5(white_water)
    'no_water_classes': [1, 3],   # original 2(sand), 4(wet_sand)
    'ignore_classes': [0, 255]    # original 1(clouds), 255(unannotated)
}

print(f"Device: {DEVICE}")
print(f"Boundary offset: {BOUNDARY_OFFSET_PIXELS} pixels")
print(f"Minimum boundary pixels: {MIN_BOUNDARY_PIXELS}")

## Initialize Dataset and DataLoader
Prepare the Test Dataset DataLoader that will fetch patches from the test split.

In [ ]:
# Initialize Dataset and DataLoader
test_dataset = CoastlineDataset(split_dir=TEST_NPY_DIR, transform=None)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Loaded {len(test_dataset)} test patches.")
# Make sure we have metadata that maps correctly to evaluation output
print(f"Dataset has metadata columns: {list(test_dataset.metadata.columns)}")

## Inference and Evaluate Boundary Metrics
Iterate through the test patches, output predictions, compare with ground truth bounding zone and aggregate the metrics using pandas.

In [ ]:
def run_evaluation_loop(test_loader, eval_models):
    results = []

    for model_config in eval_models:
        model_filepath = model_config["filepath"]
        if not model_filepath.exists():
            print(f"Skipping {model_config['name']}: {model_filepath} not found.")
            continue
            
        print(f"\n============================================================")
        print(f"Evaluating Model: {model_config['name']}")
        print(f"============================================================")
        
        # Instantiate Model
        model = get_model(arch=model_config["arch"], encoder_name=model_config["encoder"])
        
        # Load weights explicitly
        state_dict = torch.load(model_filepath, map_location=DEVICE, weights_only=True)
        model.load_state_dict(state_dict)
        model.to(DEVICE)
        model.eval()

        with torch.no_grad():
            for dt_idx, (image_tensor, gt_mask_tensor) in enumerate(tqdm(test_loader, desc=f"Evaluating Patches")):
                image_tensor = image_tensor.float().to(DEVICE)
                
                # Model inference prediction
                logits = model(image_tensor)
                # Squeeze the batch dimension because BATCH_SIZE=1 
                # Model yields (BS, Classes, H, W)
                pred_mask = torch.argmax(logits, dim=1).cpu().numpy().squeeze(0)  # Shape (128,128)
                gt_mask = gt_mask_tensor.numpy().squeeze(0)                       # Shape (128,128)
                
                # Fetch row metadata mapped directly from index
                row = test_dataset.metadata.iloc[dt_idx]
                
                # We need to collapse to binary using standard project function
                # Note: `collapse_gt_to_binary` supports the mapped numpy index
                gt_binary, ignore_mask = collapse_gt_to_binary(
                    gt_mask,
                    water_classes=MAPPED_GT_CLASS_MAPPING['water_classes'],
                    no_water_classes=MAPPED_GT_CLASS_MAPPING['no_water_classes'],
                    ignore_classes=MAPPED_GT_CLASS_MAPPING['ignore_classes']
                )

                # Prediction does not have "Ignore" values itself, it forces a decision 
                # so we can use the same utility by skipping ignore parsing outputs 
                pred_binary, _ = collapse_gt_to_binary(
                    pred_mask,
                    water_classes=MAPPED_GT_CLASS_MAPPING['water_classes'],
                    no_water_classes=MAPPED_GT_CLASS_MAPPING['no_water_classes'],
                    ignore_classes=[0, 255] # Explicit ignore doesn't matter for prediction usually
                )

                # Create boundary zone from Ground Truth
                boundary_zone = create_boundary_zone_mask(
                    gt_binary, 
                    ignore_mask, 
                    offset_pixels=BOUNDARY_OFFSET_PIXELS
                )
                
                # Evaluation mask inside the boundary that ignores NO data
                evaluation_mask = boundary_zone & (~ignore_mask)
                combined_ignore_mask = ~evaluation_mask
                
                boundary_pixel_count = np.sum(evaluation_mask)
                if boundary_pixel_count < MIN_BOUNDARY_PIXELS:
                    continue  # We skip this patch because it lacks sufficient boundary area
                
                # Calculate bounding dice logic
                metrics = calculate_metrics(gt_binary, pred_binary, combined_ignore_mask)
                
                if metrics['valid_pixel_count'] == 0:
                    continue
                
                # Parse timestamp into Date and Time to match format
                dt_obj = pd.to_datetime(row['timestamp'])
                
                # Map to standard result format
                results.append({
                    'study_name': row['study_name'],
                    'date': dt_obj.strftime("%Y-%m-%d"),
                    'time': dt_obj.strftime("%H-%M-%S").replace(':', '-'),
                    'tile': '30TVP', # Default inference tile string if not mapped
                    'site': row['study_name'].upper(),
                    'patch_id': dt_idx, # ID
                    'model': model_config['name'],
                    'arch': model_config['arch'],
                    'encoder': model_config['encoder'],
                    'iou': metrics['iou'],
                    'dice': metrics['dice'],
                    'boundary_pixel_count': int(boundary_pixel_count),
                    'boundary_offset_pixels': BOUNDARY_OFFSET_PIXELS
                })

    return pd.DataFrame(results)

df_results = run_evaluation_loop(test_loader, EVAL_MODELS)
print(f"Evaluated {len(df_results)} patches within the boundary zone.")
display(df_results.head(10))

## Summary Statistics and Visualizations
Print descriptive statistics for the U-Net boundaries. Compare metrics directly.

In [ ]:
# Ensure results output directory
output_dir = BASE_DIR / ".." / "results"
output_dir.mkdir(parents=True, exist_ok=True)

# Save inference
output_file = output_dir / "dl_boundary_metrics_results.csv"
df_results.to_csv(output_file, index=False)
print(f"Deep learning evaluation results saved to: {output_file}")

# Print summary stats
def print_summary_stats(df, title):
    print(f"--- {title} ---")
    print(f"Count (Valid Patches): {len(df)}")
    print(f"Mean IoU: {df['iou'].mean():.4f}")
    print(f"Mean Dice: {df['dice'].mean():.4f}")
    print("-------------------\\n")

print_summary_stats(df_results, "Deep Learning Segmentation (Boundary Zone)")

for model_name in df_results['model'].unique():
    model_df = df_results[df_results['model'] == model_name]
    print(f"\nModel: {model_name}")
    print(model_df.groupby('study_name')[['iou', 'dice']].mean())

# Plotting metrics
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_results.melt(
        id_vars=['study_name', 'model', 'patch_id'], 
        value_vars=['iou', 'dice'],
        var_name='metric', 
        value_name='score'
    ), 
    x='metric', 
    y='score', 
    hue='study_name', 
    split=True, 
    inner="quartile"
)

plt.title('Distribution of IoU and Dice for DL Segmentation Models (Boundary Zone)')
plt.ylabel('Metric Score')
plt.ylim(0, 1)
plt.legend(title='Study Area', loc='lower right')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()

# Save figure
plot_output_file = output_dir / "dl_evaluation_overview.png"
plt.savefig(plot_output_file, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {plot_output_file}")
plt.show()